In [ ]:
import pandas as pd
import requests

# 1. Dicionário com o código da série no SGS e o nome amigável da variável
series_bcb = {
    '21082': 'inadimplencia_total',
    '21083': 'inadimplencia_pf',
    '21084': 'inadimplencia_pj',
    '20716': 'taxa_juros_pf',
    '20717': 'taxa_juros_pj',
    '20622': 'saldo_carteira_milhoes'
}

df_list = []

# 2. Requisição para a API do SGS/BCB
for codigo, nome_coluna in series_bcb.items():
    url = f'https://api.bcb.gov.br/dados/serie/bcdata.sgs.{codigo}/dados?formato=json'
    res = requests.get(url)
    
    if res.status_code == 200:
        temp_df = pd.DataFrame(res.json())
        temp_df['data'] = pd.to_datetime(temp_df['data'], format='%d/%m/%Y')
        temp_df['valor'] = pd.to_numeric(temp_df['valor'], errors='coerce')
        temp_df = temp_df.rename(columns={'valor': nome_coluna})
        df_list.append(temp_df.set_index('data'))

# 3. Consolidação em uma única Tabela Fato
df_risco_credito = pd.concat(df_list, axis=1).sort_index()

# Filtrando os últimos 10 anos para manter o painel atualizado e leve
df_risco_credito = df_risco_credito.loc['2015-01-01':].reset_index()

# 4. Tratamento adicional (prazos, preenchimento de nulos caso haja defasagem no BCB)
df_risco_credito.ffill(inplace=True)

# 5. Exportação do dataset tratado para o Power BI
df_risco_credito.to_csv('base_risco_credito_bcb.csv', index=False, sep=';', decimal=',')

print("Extração e tratamento concluídos com sucesso!")
print(df_risco_credito.tail())